In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, concatenate, UpSampling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical


In [ ]:
# U-Net modeli oluşturma
def unet_model(input_size=(256, 256, 1), n_classes=2):
    inputs = Input(input_size)
    
    # Encoder (Downsampling path)
    c1 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(inputs)
    c1 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1)
    
    c2 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p1)
    c2 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2)
    
    c3 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p2)
    c3 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c3)
    p3 = MaxPooling2D((2, 2))(c3)
    
    c4 = Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p3)
    c4 = Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c4)
    p4 = MaxPooling2D(pool_size=(2, 2))(c4)
    
    # Bridge
    c5 = Conv2D(1024, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p4)
    c5 = Conv2D(1024, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c5)
    
    # Decoder (Upsampling path)
    u6 = UpSampling2D(size=(2, 2))(c5)
    u6 = concatenate([u6, c4])
    c6 = Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u6)
    c6 = Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c6)
    
    u7 = UpSampling2D(size=(2, 2))(c6)
    u7 = concatenate([u7, c3])
    c7 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u7)
    c7 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c7)
    
    u8 = UpSampling2D(size=(2, 2))(c7)
    u8 = concatenate([u8, c2])
    c8 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u8)
    c8 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c8)
    
    u9 = UpSampling2D(size=(2, 2))(c8)
    u9 = concatenate([u9, c1], axis=3)
    c9 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u9)
    c9 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c9)
    
    # Sınıflandırma katmanı
    outputs = Conv2D(n_classes, (1, 1), activation='softmax')(c9)
    
    model = Model(inputs=[inputs], outputs=[outputs])
    model.compile(optimizer=Adam(lr=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model


In [ ]:
# Veri ön işleme fonksiyonları
def preprocess_image(image_path, target_size=(256, 256)):
    img = load_img(image_path, color_mode='grayscale', target_size=target_size)
    img = img_to_array(img)
    img = img / 255.0  # Normalizasyon
    return img


In [ ]:

def preprocess_mask(mask_path, target_size=(256, 256), n_classes=2):
    mask = load_img(mask_path, color_mode='grayscale', target_size=target_size)
    mask = img_to_array(mask)
    mask = mask / 255.0
    mask = np.round(mask)  # İkili maske için yuvarlama
    mask = to_categorical(mask, n_classes)  # One-hot encoding
    return mask


In [ ]:
# Veri yükleme ve ön işleme
def load_data(image_dir, mask_dir, target_size=(256, 256), n_classes=2):
    image_files = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith('.png')])
    mask_files = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir) if f.endswith('.png')])
    
    images = np.array([preprocess_image(f, target_size) for f in image_files])
    masks = np.array([preprocess_mask(f, target_size, n_classes) for f in mask_files])
    
    # Yeniden şekillendirme
    images = np.reshape(images, (images.shape[0], target_size[0], target_size[1], 1))
    
    return images, masks

In [ ]:



# Model eğitimi
def train_model(images, masks, val_split=0.2, epochs=50, batch_size=8):
    print("Eğitim verisi boyutu:", images.shape)
    print("Maske verisi boyutu:", masks.shape)
    # Veri setini eğitim ve doğrulama olarak bölme
    X_train, X_val, y_train, y_val = train_test_split(images, masks, test_size=val_split, random_state=42)
    
    # Model oluşturma
    model = unet_model(input_size=(X_train.shape[1], X_train.shape[2], 1))
    
    # Callbacks
    checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_loss')
    early_stopping = EarlyStopping(patience=10, monitor='val_loss')
    
    # Eğitim
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=[checkpoint, early_stopping]
    )
    
    return model, history

In [ ]:




# Tahmin ve görselleştirme
def predict_and_visualize(model, image_path, target_size=(256, 256)):
    # Görüntüyü yükleme ve ön işleme
    img = preprocess_image(image_path, target_size)
    img = np.reshape(img, (1, target_size[0], target_size[1], 1))
    
    # Tahmin
    pred = model.predict(img)[0]
    pred_mask = np.argmax(pred, axis=-1)
    
    # Görselleştirme
    plt.figure(figsize=(12, 6))
    plt.subplot(131)
    plt.imshow(img[0, :, :, 0], cmap='gray')
    plt.title('Orijinal BT Görüntüsü')
    
    plt.subplot(132)
    plt.imshow(pred_mask, cmap='jet')
    plt.title('Kanama Tahmini')
    
    plt.subplot(133)
    # Orijinal görüntü üzerine tahmin maskesini bindirme
    overlay = np.zeros((target_size[0], target_size[1], 3))
    overlay[:, :, 0] = img[0, :, :, 0]  # Gri kanal
    overlay[:, :, 1] = img[0, :, :, 0]  # Gri kanal
    overlay[:, :, 2] = img[0, :, :, 0]  # Gri kanal
    
    # Kanama alanlarını kırmızı ile işaretleme
    overlay[pred_mask == 1, 0] = 1.0  # Kırmızı kanal
    overlay[pred_mask == 1, 1] = 0.0  # Yeşil kanal
    overlay[pred_mask == 1, 2] = 0.0  # Mavi kanal
    
    plt.imshow(overlay)
    plt.title('Bindirme Görüntüsü')
    
    plt.tight_layout()
    plt.savefig('hemorrhage_detection_result.png')
    plt.show()
    
    return pred_mask

In [ ]:
google_drive_base_path = "/content/gdrive/MyDrive/datasets/"


In [ ]:
 # Veri dizinleri
image_dir = google_drive_base_path + "Kanama Veri Seti/PNG/"
mask_dir =google_drive_base_path + "Kanama Veri Seti/Mask/kanama/"
import os
# Yeterli görüntü var mı kontrol etme
if len(os.listdir(image_dir)) > 0 and len(os.listdir(mask_dir)) > 0:
    # Veri yükleme
    images, masks = load_data(image_dir, mask_dir)
    
    # Model eğitimi
    model, history = train_model(images, masks, epochs=50)
    
    # Eğitim geçmişini görselleştirme
    plt.figure(figsize=(12, 4))
    plt.subplot(121)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model Kaybı')
    plt.ylabel('Kayıp')
    plt.xlabel('Epoch')
    plt.legend(['Eğitim', 'Doğrulama'], loc='upper right')
    
    plt.subplot(122)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model Doğruluğu')
    plt.ylabel('Doğruluk')
    plt.xlabel('Epoch')
    plt.legend(['Eğitim', 'Doğrulama'], loc='lower right')
    plt.savefig('training_history.png')
    plt.show()
    
    # Test görüntüsü üzerinde tahmin
    test_image = os.path.join(image_dir, os.listdir(image_dir)[0])
    predict_and_visualize(model, test_image)
else:
    print("Veri dizinleri boş. Lütfen görüntüleri ve maskeleri ilgili dizinlere yükleyin.")


In [ ]:

# Yeni bir görüntü üzerinde kanama tespiti yapma fonksiyonu
def detect_hemorrhage(model_path, image_path):
    """
    Önceden eğitilmiş model ile bir beyin BT görüntüsünde kanama tespiti yapar
    
    Args:
        model_path: Eğitilmiş model dosyasının yolu
        image_path: Analiz edilecek BT görüntüsünün yolu
        
    Returns:
        hemorrhage_mask: Kanama bölgelerini gösteren maske
        hemorrhage_percentage: Kanamanın beyin hacmine oranı
    """
    from tensorflow.keras.models import load_model
    import cv2
    
    # Modeli yükleme
    model = load_model(model_path)
    
    # Görüntüyü okuma ve ön işleme
    img = preprocess_image(image_path)
    img = np.reshape(img, (1, img.shape[0], img.shape[1], 1))
    
    # Tahmin
    pred = model.predict(img)[0]
    hemorrhage_mask = np.argmax(pred, axis=-1)
    
    # Beyin bölgesini tespit etme (basit eşikleme ile)
    original_img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    original_img = cv2.resize(original_img, (256, 256))
    _, brain_mask = cv2.threshold(original_img, 10, 255, cv2.THRESH_BINARY)
    brain_area = np.sum(brain_mask > 0)
    
    # Kanama bölgesinin alanı
    hemorrhage_area = np.sum(hemorrhage_mask == 1)
    
    # Kanamanın oranı
    hemorrhage_percentage = (hemorrhage_area / brain_area) * 100 if brain_area > 0 else 0
    
    return hemorrhage_mask, hemorrhage_percentage